#**Goal: Create a code explanation for each cell as text below it.**

**Creating a hybrid search system using**
* Embeddings for semantic search (sentence_transformers)
* BM25 for keyword ranking (Sparse retrieval)
* FAISS as a index.









In [1]:
!pip install sentence-transformers


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install rank_bm25


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!pip install faiss-cpu


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import sentence_transformers

c:\Users\mithsuka.dikkumbura\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import faiss

"documents" is a small document collection. Hybrid search will rank these documents base on both keyword 

"query" is the user queary

In [6]:
documents = [
    "Artificial Intelligence is changing the world.",
    "Machine Learning is a subset of AI.",
    "Deep Learning is a subset of Machine Learning.",
    "Natural Language Processing involves understanding text.",
    "Computer Vision allows machines to see and understand.",
    "AI includes areas like NLP and Computer Vision.",
    "The Pyramids of Giza are architectural marvels.",
    "Mozart was a prolific composer during the classical era.",
    "Mount Everest is the tallest mountain on Earth.",
    "The Nile is one of the world's longest rivers.",
    "Van Gogh's Starry Night is a popular piece of art.",
    "Basketball is a sport played with a round ball and two teams."
]

In [7]:
query = "Tell me about AI in text and vision."

Split each document in to words and store them as a list

In [8]:
tokenized_corpus = [doc.split(" ") for doc in documents]

Create BM25. This prepares the system to score documents for keyword relevance

In [9]:
bm25 = BM25Okapi(tokenized_corpus)

Load model ('paraphrase-MiniLM-L6-v2') converts text into dense vectors

Genarate embeddings for every document.Each document become a numeric vector we can compare in semantic search

In [10]:
model = SentenceTransformer('paraphrase-MiniLM-L6-v2')

c:\Users\mithsuka.dikkumbura\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [11]:
document_embeddings = model.encode(documents)

Create FAISS index using L2 distance. The index dimension must match the embedding vector size

In [12]:
index = faiss.IndexFlatL2(document_embeddings.shape[1])

Add all document embeddings into FAISS

In [13]:
index.add(np.array(document_embeddings).astype('float32'))


Define how many top documents want to keep after BM25 filtering

In [14]:
top_n =10

Computes BM25 keyword relevance scores for every document using the query tokens

In [15]:
bm25_scores = bm25.get_scores(query.split(" "))

Select indices of the top BM25 documents. This reduce the search space before semantic re-ranking

In [16]:
top_docs_indices = np.argsort(bm25_scores)[-top_n:]

Collects embeddings only for the BM25-selected documents. These are the candidates that will be re-ranked using semantic similarity.

In [17]:
top_docs_embeddings = [document_embeddings[i] for i in top_docs_indices]

Converts the query into an embedding vector so we can compare it against candidate document vectors

In [18]:
query_embedding = model.encode([query])

Creates a new FAISS index only for the BM25 candidate embeddings.
This makes the semantic re-ranking step faster and focused.

In [19]:
sub_index = faiss.IndexFlatL2(top_docs_embeddings[0].shape[0])

Add the BM25 candidate embeddings into the smaller FAISS index for semantic search.

In [20]:
sub_index.add(np.array(top_docs_embeddings).astype('float32'))

Search inside the BM25 candidates using the query embedding and returns the most semantically similar results.

In [21]:
_,sub_dense_ranked_indices = sub_index.search(np.array(query_embedding).astype('float32'), top_n)

Shows the ranked positions inside the BM25 candidate list 

In [22]:
sub_dense_ranked_indices


array([[9, 8, 1, 0, 6, 7, 2, 4, 3, 5]])

Maps the candidate-level FAISS results back to the original document indices.

In [23]:
final_ranked_indices = [top_docs_indices[i] for i in sub_dense_ranked_indices[0]]

Uses the final ranked indices to fetch the actual document strings in the final ranked order.

In [24]:
ranked_docs = [documents[i] for i in final_ranked_indices]

Output the final list of ranked documents

In [25]:
ranked_docs

['AI includes areas like NLP and Computer Vision.',
 'Computer Vision allows machines to see and understand.',
 'Natural Language Processing involves understanding text.',
 'Deep Learning is a subset of Machine Learning.',
 "Van Gogh's Starry Night is a popular piece of art.",
 'Basketball is a sport played with a round ball and two teams.',
 'Mozart was a prolific composer during the classical era.',
 "The Nile is one of the world's longest rivers.",
 'The Pyramids of Giza are architectural marvels.',
 'Mount Everest is the tallest mountain on Earth.']

#Provide a brief description of the process this code implements.

This notebook implements a hybrid search pipeline. It uses BM25 to find documents that match the query keywords and keeps the top candidates. Then it converts the query and those candidates into embeddings using a SentenceTransformer model. Finally use FAISS to re-rank the BM25 candidates by semantic similarity, producing results that match both keywords and meaning.